In [ ]:
import sys
sys.path.append('../src')
sys.path.append('../data')
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import pandas as pd
from seir_model.simulator import SEIRSimulator

In [ ]:
# --- Load data and posterior ---
data = pd.read_csv("data/processed_epidemic_data.csv")
population = 1_000_000
trace = az.from_netcdf("results/trace.nc")
days = len(data)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta,0.299,0.102,0.112,0.484,0.002,0.002,2201.0,1677.0,1.0
sigma,0.201,0.099,0.019,0.395,0.002,0.002,2759.0,1724.0,1.0
gamma,0.102,0.103,-0.088,0.297,0.002,0.002,2850.0,1637.0,1.0
hosp_rate,0.050,0.010,0.032,0.070,0.000,0.000,2846.0,1422.0,1.0
mu,0.020,0.010,0.001,0.038,0.000,0.000,2530.0,1816.0,1.0
p_sym,0.601,0.058,0.507,0.695,0.001,0.001,2660.0,1358.0,1.0
p_test,0.556,0.260,0.156,0.998,0.005,0.004,2390.0,1361.0,1.0


In [ ]:
# --- Extract posterior samples ---
posterior = trace.posterior
n_draws = posterior.dims["draw"]

,S,E,I,Sym,Asym,Hosp,R,D,day,simulation
0,999990.000000,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0
1,999989.015948,4.299030,1.685022,3.515086,1.484914,0.000000,0.000000,0.000000,1,0
2,999988.689832,3.169794,1.455352,4.150108,1.843969,0.163374,0.527571,0.000000,2,0
3,999988.404110,2.373945,1.081571,4.540496,2.090647,0.355516,1.315584,0.000752,3,0
4,999988.193196,1.777088,0.807771,4.611115,2.204574,0.562249,2.358075,0.002412,4,0


In [ ]:
# --- Prepare for simulation ---
simulated_trajectories = []

for draw_idx in range(n_draws):
    draw = {var: posterior[var].values[0, draw_idx] for var in [
        "beta", "incubation", "symptomatic",
        "h_rate", "h_duration", "icu_rate", "icu_duration", "mu",
        "E0", "I0"
    ]}

    params = {
        "beta": draw["beta"],
        "incubation": draw["incubation"],
        "symptomatic": draw["symptomatic"],
        "h_rate": draw["h_rate"],
        "h_duration": draw["h_duration"],
        "icu_rate": draw["icu_rate"],
        "icu_duration": draw["icu_duration"],
        "mu": draw["mu"]
    }

In [ ]:
 E0, I0 = int(draw["E0"]), int(draw["I0"])
    S0 = population - E0 - I0
    initial_state = [S0, E0, I0, 0, 0, 0, 0]

    simulator = SEIRSimulator(N=population, params=params, stochastic=True)
    sim = simulator.simulate(initial_state, days)

    simulated_trajectories.append(sim)

# --- Convert to array: [draws, days, compartments] ---
sim_array = np.stack(simulated_trajectories)


In [ ]:
# --- Compute credible intervals ---
compartments = ["S", "E", "I", "H", "ICU", "R", "D"]
sim_stats = {}

for i, name in enumerate(compartments):
    comp_samples = sim_array[:, :, i]
    sim_stats[name] = {
        "mean": np.mean(comp_samples, axis=0),
        "hdi_3%": np.percentile(comp_samples, 3, axis=0),
        "hdi_97%": np.percentile(comp_samples, 97, axis=0)
    }

In [ ]:
# --- Plot: ICU with 94% interval ---
plt.figure(figsize=(10, 5))
plt.plot(data["Day"], data["num_critical"], "o", label="Observed ICU")
plt.plot(data["Day"], sim_stats["ICU"]["mean"], label="Predicted ICU")
plt.fill_between(
    data["Day"],
    sim_stats["ICU"]["hdi_3%"],
    sim_stats["ICU"]["hdi_97%"],
    alpha=0.3,
    label="94% credible interval"
)
plt.xlabel("Day")
plt.ylabel("Number of ICU patients")
plt.title("Posterior Predictive Simulation")
plt.legend()
plt.tight_layout()
plt.show()


: 